# 01 - Hola Ollama

En este notebook aprenderemos a:
- Conectarnos a Ollama desde Python
- Enviar prompts y recibir respuestas
- Usar streaming para ver la respuesta en tiempo real
- Entender los parámetros básicos de generación

## 1.1 Verificar que Ollama está corriendo

Antes de empezar, asegurémonos de que Ollama está activo y tiene modelos descargados.

In [1]:
import requests

# Verificar conexión con Ollama
try:
    response = requests.get("http://localhost:11434/api/tags")
    modelos = response.json()["models"]
    print("Ollama está corriendo. Modelos disponibles:")
    for m in modelos:
        print(f"  - {m['name']} ({m['size'] / 1e9:.1f} GB)")
except requests.ConnectionError:
    print("ERROR: Ollama no está corriendo. Ejecuta 'ollama serve' en otra terminal.")

Ollama está corriendo. Modelos disponibles:
  - gemma3:27b (17.4 GB)
  - gemma3:12b (8.1 GB)


## 1.2 Primera conexión con langchain-ollama

LangChain provee el paquete `langchain-ollama` que facilita la integración.

In [2]:
from langchain_ollama import ChatOllama

# Crear instancia del modelo
llm = ChatOllama(
    model="gemma3:12b",
    temperature=0.7,  # Creatividad (0=determinista, 1=muy creativo)
)

# Enviar un mensaje simple
respuesta = llm.invoke("¿Qué es un agente de IA? Responde en 2 oraciones.")
print(respuesta.content)

Un agente de IA es una entidad, ya sea un software o un robot, que percibe su entorno a través de sensores y actúa sobre él a través de actuadores para lograr un objetivo específico. En esencia, toma decisiones para maximizar su éxito en la consecución de sus objetivos.



## 1.3 Streaming (respuesta en tiempo real)

Con streaming vemos la respuesta token por token, como en ChatGPT.

In [3]:
# Streaming: ver la respuesta mientras se genera
for chunk in llm.stream("Explica qué es LangChain en 3 oraciones."):
    print(chunk.content, end="", flush=True)
print()  # Nueva línea al final

LangChain es un framework para desarrollar aplicaciones impulsadas por modelos de lenguaje grandes (LLMs). Permite conectar LLMs con otras fuentes de datos y herramientas, facilitando la creación de aplicaciones más complejas como chatbots, agentes y pipelines de procesamiento de lenguaje natural. En esencia, LangChain simplifica la construcción de aplicaciones con IA conversacional y automatizada.



## 1.4 Mensajes con roles

Los modelos de chat entienden diferentes roles: `system`, `human` y `ai`.

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage

mensajes = [
    SystemMessage(content="Eres un asistente experto en historia de Colombia. Responde de forma concisa."),
    HumanMessage(content="¿Cuándo se fundó Bogotá?"),
]

respuesta = llm.invoke(mensajes)
print(respuesta.content)

Bogotá fue fundada el 6 de agosto de 1538 por Gonzalo Jiménez de Quesada.


## 1.5 Parámetros de generación

Podemos ajustar cómo genera texto el modelo.

In [5]:
# Modelo determinista (siempre la misma respuesta)
llm_determinista = ChatOllama(model="gemma3:12b", temperature=0)

# Modelo creativo
llm_creativo = ChatOllama(model="gemma3:12b", temperature=1.0)

pregunta = "Inventa un nombre para una tienda de café colombiano"

print("=== Determinista (temperature=0) ===")
for i in range(3):
    r = llm_determinista.invoke(pregunta)
    print(f"  Intento {i+1}: {r.content[:80]}")

print("\n=== Creativo (temperature=1.0) ===")
for i in range(3):
    r = llm_creativo.invoke(pregunta)
    print(f"  Intento {i+1}: {r.content[:80]}")

=== Determinista (temperature=0) ===
  Intento 1: ¡Claro! Aquí te dejo algunas ideas para el nombre de una tienda de café colombia
  Intento 2: ¡Claro! Aquí te dejo algunas ideas para el nombre de una tienda de café colombia
  Intento 3: ¡Claro! Aquí te dejo algunas ideas para el nombre de una tienda de café colombia

=== Creativo (temperature=1.0) ===
  Intento 1: ¡Claro! Aquí te dejo algunas ideas para el nombre de una tienda de café colombia
  Intento 2: ¡Claro! Aquí tienes algunas ideas para el nombre de una tienda de café colombian
  Intento 3: ¡Claro! Aquí te dejo algunas ideas de nombres para una tienda de café colombiano


## Ejercicio

1. Cambia el modelo a `gemma3:27b` (si lo tienes descargado) y compara las respuestas
2. Experimenta con `temperature` valores de 0, 0.5 y 1.0
3. Crea un SystemMessage que haga al modelo responder siempre en inglés

In [6]:
# Tu código aquí
